In this section, we will be working with the UNSW-NB15 dataset, a widely used benchmark dataset for network intrusion and anomaly detection. Here rather than using merely the volume of traffic we will be trying to identify attacks with a high degree of accuracy and speed using other features of the requests and their origins.

The goal of this assignment is to explore how machine learning models can be used to detect attacks with both high accuracy and low latency. In particular, we will focus on gradient boosting methods, which are well-suited for tabular data and can capture complex nonlinear relationships between features. You will implement and evaluate a LightGBM-based model, examining how it performs relative to alternative approaches in terms of predictive power and computational efficiency.


This exercise is inspired by [this](https://ieeexplore.ieee.org/document/9315049) paper, which tests LightGBM's performance relative to its alternatives.

## Setup and Imports

In [ ]:
import json
import time
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
# Load the data from Google Drive
from google.colab import drive
drive.mount('/content/drive/')  #you may be prompted to change the permissions on your google drive account here, just hit 'select all'

RANDOM_STATE = 808
TEST_SIZE = 0.2

DATA_PATH = None
FEATURE_PATH = None

## Load Data and Assign Canonical Column Names

In [ ]:
feature_df = pd.read_csv(FEATURE_PATH, encoding='cp1252')
feature_names = feature_df["Name"].astype(str).str.strip().str.replace(" ", "", regex=False).tolist()

if len(feature_names) != 49:
    raise ValueError(f"Expected 49 feature names, got {len(feature_names)}")

df = pd.read_csv(DATA_PATH, header=None)
if df.shape[1] != len(feature_names):
    raise ValueError(f"Data has {df.shape[1]} columns, but schema has {len(feature_names)}")

df.columns = feature_names
print(f"Loaded shape: {df.shape}")
print(f"Columns assigned: {len(df.columns)}")
df.head(2)

## Preprocessing and Feature Construction

In [ ]:
df = df.copy()

# TODO: revisit preprocessing choices against official UNSW feature engineering notes.
# Normalize targets first.
df["attack_cat"] = (
    df["attack_cat"]
    .fillna("Normal")
    .astype(str)
    .str.strip()
    .replace({"": "Normal", "-": "Normal"})
)
df["Label"] = pd.to_numeric(df["Label"], errors="coerce").fillna(0).astype(int)

# Drop leakage/identifier fields from modeling features.
drop_for_modeling = ["srcip", "dstip", "Stime", "Ltime"]
drop_for_modeling = [c for c in drop_for_modeling if c in df.columns]

X_raw = df.drop(columns=drop_for_modeling + ["Label", "attack_cat"])
y_binary = df["Label"]
y_multi_raw = df["attack_cat"]


#TODO: Specify the categorical columns we have in the dataset

#TODO: Preprocess some of your features (hint: see the linked paper)




X = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=False)
print(f"Model feature matrix shape after one-hot encoding: {X.shape}")

In [ ]:
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X,
    y_binary,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_binary,
)

print("Binary split shapes:")
print("X_train:", X_train_bin.shape, "X_test:", X_test_bin.shape)
print("y_train mean:", y_train_bin.mean(), "y_test mean:", y_test_bin.mean())
print("Binary train distribution:")
print(y_train_bin.value_counts(normalize=True).sort_index())

XGBoost Baseline Model (you shouldn't need to edit the cell below)

In [ ]:
neg_count = int((y_train_bin == 0).sum())
pos_count = int((y_train_bin == 1).sum())
scale_pos_weight = (neg_count / pos_count) if pos_count > 0 else 1.0

xgb_model = XGBClassifier(
    random_state=RANDOM_STATE,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=1.0,
    colsample_bytree=1.0,
)

t0 = time.perf_counter()
xgb_model.fit(X_train_bin, y_train_bin)
xgb_train_seconds = time.perf_counter() - t0

t1 = time.perf_counter()
xgb_preds = xgb_model.predict(X_test_bin)
xgb_infer_seconds = time.perf_counter() - t1

xgb_infer_full_est_seconds = xgb_infer_seconds * (len(X) / len(X_test_bin))
xgb_total_load_test_split = xgb_train_seconds + xgb_infer_seconds
xgb_total_load_full_est = xgb_train_seconds + xgb_infer_full_est_seconds
xgb_acc = accuracy_score(y_test_bin, xgb_preds)

print(
    "XGBoost (scale_pos_weight): "
    f"train={xgb_train_seconds:.4f}s, infer={xgb_infer_seconds:.4f}s, "
    f"total_load={xgb_total_load_test_split:.4f}s, acc={xgb_acc:.4f}"
)

xgb_result = {
    "Model": "XGBoost (scale_pos_weight)",
    "Train Time (s)": xgb_train_seconds,
    "Inference Time (s)": xgb_infer_seconds,
    "Total Load (s)": xgb_total_load_test_split,
    "Estimated Full Inference Load (s)": xgb_infer_full_est_seconds,
    "Estimated Full Total Load (s)": xgb_total_load_full_est,
    "Binary Accuracy": xgb_acc,
    "Imbalance Handling": f"scale_pos_weight={scale_pos_weight:.4f}",
}

## 6) Binary LightGBM Models: Vanilla vs Bundled vs GOSS

In [ ]:
# TODO: Fill in this wrapper function to run LGB Models, generate predictions, time and score.
def run_binary_experiment(model_name, model_kwargs):
    # Tell LightGBM this binary task is imbalanced.
    binary_imbalance_kwargs = {"is_unbalance": True}

    # TODO: Create a lightGBM model that can take in different hyperparameters
    model=None


    t0 = time.perf_counter()
    # TODO: Fit the model




    train_seconds = time.perf_counter() - t0

    t1 = time.perf_counter()
    # TODO: Generate predictions
    preds = None



    infer_seconds = time.perf_counter() - t1

    # Estimated full-data inference load assumes linear scaling with row count.
    infer_full_est_seconds = infer_seconds * (len(X) / len(X_test_bin))
    total_load_test_split = train_seconds + infer_seconds
    total_load_full_est = train_seconds + infer_full_est_seconds

    acc = accuracy_score(y_test_bin, preds)
    print(
        f"{model_name}: train={train_seconds:.4f}s, infer={infer_seconds:.4f}s, "
        f"total_load={total_load_test_split:.4f}s, acc={acc:.4f}"
    )

    return {
        "Model": model_name,
        "Train Time (s)": train_seconds,
        "Inference Time (s)": infer_seconds,
        "Total Load (s)": total_load_test_split,
        "Estimated Full Inference Load (s)": infer_full_est_seconds,
        "Estimated Full Total Load (s)": total_load_full_est,
        "Binary Accuracy": acc,
        "Imbalance Handling": "is_unbalance=True",
    }


binary_results = [xgb_result]

##TODO: Duplicate the functions below and adjust the settings to test the model with the following settings:
##Model 2: Use Exclusive Feature Bundling
##Model 3: Use GOSS as a sampling strategy
##Model 4: Sample by bagging instead, use a 50% sample
binary_results.append(run_binary_experiment("Vanilla (enable_bundle=False)", {"enable_bundle": False,}))
#binary_results.append(run_binary_experiment())


# Comparison Table:

In [ ]:
comparison_df = pd.DataFrame(binary_results)
comparison_df = comparison_df.sort_values(by="Binary Accuracy", ascending=False).reset_index(drop=True)
comparison_df = comparison_df[
    [
        "Model",
        "Train Time (s)",
        "Inference Time (s)",
        "Total Load (s)",
        "Estimated Full Inference Load (s)",
        "Estimated Full Total Load (s)",
        "Binary Accuracy",
        "Imbalance Handling",
    ]
]
comparison_df

Higher binary accuracy indicates better predictive quality, while lower train/inference times indicate better efficiency. How does LightGBM compare with XGBoost in terms of accuracy in this case? How about speed?

What is the argument for why the GOSS sampling strategy should yield speed gains? How does this strategy go about maintaining accuracy? Does GOSS achieve the speed gains we would expect in this case? If no, why might that might be (hint: think about the steps required by that sampling algorithm)?

Explain in a couple sentences how the exclusive feature bundling algorithm decides what features to bundle. Why does bundling features accelerate model training?

It would be helpful for your cybersecurity operation to not just be able to identify whether or not there is an anomaly but also what *type* of attack is occurring. Luckily we can also use the LightGBM model on nonbinary classification problems.



In [ ]:
# TODO: Create a classifier that predicts different types of system attacks.
label_encoder = LabelEncoder()
y_multi = label_encoder.fit_transform(y_multi_raw)






##TODO: Generate classification metrics that correspond to the following
##Use classification_report()
multi_metrics = {
    "Model": "Multiclass LGBM",
    "Train Time (s)": multi_train_seconds,
    "Inference Time (s)": multi_infer_seconds,
    "Total Load (s)": multi_total_load_test_split,
    "Estimated Full Inference Load (s)": multi_infer_full_est_seconds,
    "Estimated Full Total Load (s)": multi_total_load_full_est,
    "Multiclass Accuracy": multi_acc,
    "Imbalance Handling": "class_weight='balanced'",
    "Num Classes": len(label_encoder.classes_),
}

print("\nCompact class-wise report:")
print(classification_report(y_test_multi, multi_preds, target_names=label_encoder.classes_, zero_division=0))

## Export All Measurements to JSON

In [ ]:
##TODO: Output training statistics to json
##If you've stored results in comparison_df and multi_metrics you shouldn't have to edit this cell.
output_payload = {
    "dataset": {
        "data_path": DATA_PATH,
        "feature_schema_path": FEATURE_PATH,
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
    },
    "binary_models": comparison_df.to_dict(orient="records"),
    "multiclass_model": multi_metrics,
}

json_output_path = "unsw_nb15_lightgbm_measurements.json"
with open(json_output_path, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, indent=2)
